In [3]:
import pandas as pd
import re

def get_consecutive_indices(index_df, start_index, n_samples):
    """
    Return the next n_samples index names starting from start_index,
    based on their order in the CSV file.
    """

    index_list = index_df["Index_Name"].tolist()

    if start_index not in index_list:
        raise ValueError(f"{start_index} not found in index file.")

    start_pos = index_list.index(start_index)

    selected = index_list[start_pos:start_pos + n_samples]

    if len(selected) < n_samples:
        raise ValueError(
            f"Not enough indices available after {start_index} "
            f"for {n_samples} samples."
        )

    return selected

def generate_novaseq_samplesheet(
    index_file,
    sample_ids,
    run_name,
    output_file="SampleSheet.csv",
    start_index=None,
    manual_assignments=None,
    read1_cycles=151,
    read2_cycles=151,
    index1_cycles=10,
    index2_cycles=10,
):
    """
    Generate NovaSeq X SampleSheet.csv

    Parameters:
    ----------
    index_file : str
        Path to Illumina index set CSV.
    sample_ids : list
        List of sample IDs.
    run_name : str
        Run name for SampleSheet header.
    output_file : str
        Output CSV path.
    start_index : str
        Starting index name for automatic assignment (e.g. 'UDP0022').
    manual_assignments : dict
        Optional dict: {sample_id: index_name}
    """

    # Read index table
    index_df = pd.read_csv(index_file,index_col=False)

    required_cols = [
        "Index_Name",
        "i7_Bases_for_Sample_Sheet",
        "i5_Bases_for_Sample_Sheet_in_Forward_Orientation"
    ]

    for col in required_cols:
        if col not in index_df.columns:
            raise ValueError(f"Missing required column: {col}")

    # Build lookup
    index_lookup = index_df.set_index("Index_Name").to_dict(orient="index")
    assigned_indices = {}

    # Manual assignments first
    if manual_assignments:
        assigned_indices.update(manual_assignments)

    # Automatic continuous assignment
    if start_index:
        unassigned_samples = [s for s in sample_ids if s not in assigned_indices]

        consecutive_indices = get_consecutive_indices(
            index_df,
            start_index,
            len(unassigned_samples)
        )

        for sample, index_name in zip(unassigned_samples, consecutive_indices):
            assigned_indices[sample] = index_name

    # Validate all samples assigned
    missing = [s for s in sample_ids if s not in assigned_indices]
    if missing:
        raise ValueError(
            f"Missing index assignments for samples: {missing}"
        )

    # Build BCLConvert data rows
    rows = []
    for i, sample in enumerate(sample_ids):
        lane = (i % 8) + 1
        idx_name = assigned_indices[sample]

        rows.append({
            "Lane": lane,
            "Sample_ID": sample,
            "Index": index_lookup[idx_name]["i7_Bases_for_Sample_Sheet"],
            "Index2": index_lookup[idx_name]["i5_Bases_for_Sample_Sheet_in_Forward_Orientation"]
        })

    data_df = pd.DataFrame(rows)

    # Write SampleSheet
    with open(output_file, "w") as f:
        # Header
        f.write("[Header],\n")
        f.write("FileFormatVersion,2\n")
        f.write(f"RunName,{run_name}\n")
        f.write("InstrumentPlatform,NovaSeqXSeries\n")
        f.write("IndexOrientation,Forward\n\n")

        # Reads
        f.write("[Reads]\n")
        f.write(f"Read1Cycles,{read1_cycles}\n")
        f.write(f"Read2Cycles,{read2_cycles}\n")
        f.write(f"Index1Cycles,{index1_cycles}\n")
        f.write(f"Index2Cycles,{index2_cycles}\n\n")

        # Sequencing settings
        f.write("[Sequencing_Settings]\n")
        f.write("LibraryPrepKits,IlluminaDNAPrep\n\n")

        # BCLConvert settings
        f.write("[BCLConvert_Settings]\n")
        f.write("SoftwareVersion,4.3.16\n")
        f.write("AdapterRead1,CTGTCTCTTATACACATCT\n")
        f.write("AdapterRead2,CTGTCTCTTATACACATCT\n")
        f.write(
            f"OverrideCycles,Y{read1_cycles};I{index1_cycles};I{index2_cycles};Y{read2_cycles}\n"
        )
        f.write("FastqCompressionFormat,gzip\n")
        f.write("GenerateFastqcMetrics,true\n\n")

        # Data
        f.write("[BCLConvert_Data]\n")
        data_df.to_csv(f, index=False)

    print(f"SampleSheet written to: {output_file}")


# Example usage
if __name__ == "__main__":
    samples = [
        "WWKK2426-110",
        "WWKL2443-110",
        "WWKM2444-110",
        "WWKN2445-110"
    ]

    generate_novaseq_samplesheet(
        index_file="set_A_index_adapters.csv",
        sample_ids=samples,
        run_name="XRes52",
        start_index="UDP0052",
        output_file="SampleSheet.csv"
    )

SampleSheet written to: SampleSheet.csv


In [ ]:
#!/usr/bin/env python3
"""
NovaSeq X SampleSheet generator.

UPDATED FEATURES:
- Input file MUST contain a header.
- Supports 1, 2, or 3 columns:
    sample_name,index_name,lane
- Lane column is optional but header is required.
- If lane is provided → overrides automatic lane cycling.
- Lane must be integer 1–8.
- Wrap-around index assignment.
- Extensive validation for non-technical users.
"""

from __future__ import annotations

import argparse
import csv
import sys
from pathlib import Path
import pandas as pd
import collections
from datetime import datetime

REQUIRED_INDEX_COLUMNS = {
    "Index_Name",
    "i7_Bases_for_Sample_Sheet",
    "i5_Bases_for_Sample_Sheet_in_Forward_Orientation",
}

REQUIRED_INPUT_HEADER = ["sample_name", "index_name", "lane"]


class SampleSheetError(Exception):
    pass

LOG_MESSAGES = []

def log(message):
    print(message)
    LOG_MESSAGES.append(message)

def write_log_file(log_file):
    with open(log_file, "w") as f:
        f.write(
            f"NovaSeq SampleSheet Generator Log\n"
            f"Generated: {datetime.now()}\n\n"
        )

        for message in LOG_MESSAGES:
            f.write(message + "\n")

def load_index_table(index_file: str) -> pd.DataFrame:
    df = pd.read_csv(index_file,index_col=False)

    missing = REQUIRED_INDEX_COLUMNS - set(df.columns)
    if missing:
        raise SampleSheetError(f"Missing index columns: {missing}")

    return df

def detect_delimiter(file_path: str) -> str:
    with open(file_path, "r", newline="") as f:
        sample = f.read(2048)
        try:
            return csv.Sniffer().sniff(sample, delimiters=",\t; ").delimiter
        except Exception:
            return None

def validate_lane(value, sample):
    if pd.isna(value):
        return None

    if not str(value).isdigit():
        raise SampleSheetError(f"Lane for sample '{sample}' must be a number (1–8).")

    lane = int(value)
    if not 1 <= lane <= 8:
        raise SampleSheetError(f"Lane for sample '{sample}' must be between 1 and 8.")

    return lane


def get_wrapped_indices(index_names, start_index, n):
    if start_index not in index_names:
        raise SampleSheetError(f"Start index '{start_index}' not found.")

    start_pos = index_names.index(start_index)
    indexlist = [index_names[(start_pos + i) % len(index_names)] for i in range(n)]
    log(f"INFO: Automatic index assignment from {indexlist[0]} - {indexlist[-1]}")
    return indexlist

def validate_duplicate_indices_per_lane(df):

    duplicate_found = False

    grouped = (
        df.groupby(["lane", "index_name"])
          .size()
          .reset_index(name="count")
    )

    duplicates = grouped[grouped["count"] > 1]

    if duplicates.empty:
        return

    duplicate_found = True

    log("")
    log("ERROR: Duplicate index assignments detected within lanes:")
    log("")

    for _, row in duplicates.iterrows():

        lane = row["lane"]
        index_name = row["index_name"]
        count = row["count"]

        affected = df[
            (df["lane"] == lane)
            & (df["index_name"] == index_name)
        ]["sample_name"].tolist()

        log(
            f"Lane {lane}: {index_name} "
            f"used {count} times"
        )

        for sample in affected:
            log(f"  - {sample}")

    if duplicate_found:
        raise SampleSheetError(
            "Duplicate index assignments found within the same lane."
        )

def parse_input_file(file_path, index_names, start_index=None):
    df = pd.read_csv(file_path,sep=detect_delimiter(file_path),engine="python")

    # Validate header
    header = list(df.columns)

    if "sample_name" not in header:
        raise SampleSheetError("Input file must contain column: sample_name")

    # Normalize missing columns
    if "index_name" not in header:
        df["index_name"] = None
    if "lane" not in header:
        df["lane"] = None

    samples = df["sample_name"].tolist()
    sample_counts = collections.Counter(samples)

    duplicates = {
        sample: count
        for sample, count in sample_counts.items()
        if count > 1
    }

    if duplicates:
        log("WARNING: Duplicate sample names detected:")
        for sample, count in sorted(duplicates.items()):
            log(f"  {sample}: {count} occurrences")

    # Normalize empty strings
    df["index_name"] = df["index_name"].replace("", pd.NA)

    all_missing = df["index_name"].isnull().all()
    some_missing = df["index_name"].isnull().any()

    # -------------------------------------------------
    # CASE 1: fully automatic assignment
    # -------------------------------------------------
    if all_missing:

        if not start_index:
            raise SampleSheetError(
                "No index_name values provided. "
                "Please specify --start-index."
            )

        indices = get_wrapped_indices(
            index_names,
            start_index,
            len(df)
        )

        df["index_name"] = indices

        log(
            f"INFO: Automatically assigned indices "
            f"INFO: starting from {start_index}"
        )

    # -------------------------------------------------
    # CASE 2: hybrid assignment
    # -------------------------------------------------
    elif some_missing:

        if not start_index:
            raise SampleSheetError(
                "ERROR: Some samples contain index_name values while others are empty.\n"
                "ERROR: Please either:\n"
                "ERROR:   - provide all index_name values manually\n"
                "ERROR:   - OR use --start-index for automatic completion"
            )

        log(
            "\nWARNING: Hybrid index assignment detected.\n"
            "WARNING: Some samples already contain configured index names,\n"
            "WARNING: while others are missing.\n"
            "WARNING: Missing index names will now be automatically assigned\n"
            f"WARNING: starting from: {start_index}\n"
        )

        # Get required number of indices
        missing_count = df["index_name"].isnull().sum()

        auto_indices = get_wrapped_indices(
            index_names,
            start_index,
            missing_count
        )

        auto_iter = iter(auto_indices)

        assigned_samples = []

        for idx, row in df.iterrows():

            if pd.isna(row["index_name"]):

                new_index = next(auto_iter)

                df.at[idx, "index_name"] = new_index

                assigned_samples.append(
                    f"{row['sample_name']} -> {new_index}"
                )

        log("INFO: Automatically assigned indices:")
        for x in assigned_samples:
            log(f"INFO: - {x}")

    # -------------------------------------------------
    # CASE 3: fully manual assignment
    # -------------------------------------------------
    else:

        log("INFO: Using fully manual index assignment.")

    # Validate index names
    for s, idx in zip(df["sample_name"], df["index_name"]):
        if idx not in index_names:
            raise SampleSheetError(f"Invalid index '{idx}' for sample '{s}'.")

    # Validate lanes
    df["lane"] = [validate_lane(v, s) for v, s in zip(df["lane"], df["sample_name"])]

    return df


def write_samplesheet(output, run_name, df, index_lookup, read_cycles, index_cycles):
    with open(output, "w") as f:
        f.write("[Header],\n")
        f.write("FileFormatVersion,2\n")
        f.write(f"RunName,{run_name}\n")
        f.write("InstrumentPlatform,NovaSeqXSeries\n")
        f.write("IndexOrientation,Forward\n\n")

        f.write("[Reads]\n")
        f.write(f"Read1Cycles,{read_cycles}\n")
        f.write(f"Read2Cycles,{read_cycles}\n")
        f.write(f"Index1Cycles,{index_cycles}\n")
        f.write(f"Index2Cycles,{index_cycles}\n\n")

        f.write("[BCLConvert_Settings]\n")
        f.write("SoftwareVersion,4.3.16\n")
        f.write("AdapterRead1,CTGTCTCTTATACACATCT\n")
        f.write("AdapterRead2,CTGTCTCTTATACACATCT\n")
        f.write("OverrideCycles,Y151;I10;I10;Y151\n")
        f.write("FastqCompressionFormat,gzip\n")
        f.write("GenerateFastqcMetrics,true\n\n")

        f.write("[BCLConvert_Data]\n")
        f.write("Lane,Sample_ID,Index,Index2\n")

        for i, row in df.iterrows():
            lane = row["lane"] if row["lane"] else (i % 8) + 1
            idx = index_lookup[row["index_name"]]

            f.write(
                f"{lane},{row['sample_name']},{idx['i7_Bases_for_Sample_Sheet']},{idx['i5_Bases_for_Sample_Sheet_in_Forward_Orientation']}\n"
            )


# ---------------- CLI ----------------

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--index-file","-i", required=True, help="Illumina index set CSV (default: set_A-D_index_adapters.csv)")
    parser.add_argument("--sample-config","-c", required=True, help="Sample input file (see: README)")
    parser.add_argument("--run-name","-r", default="XResXX", help="Run name (default: XResXX)")
    parser.add_argument("--start-index","-s",
                        help="Start index for automatic assignment (required for one-column input) example: UDP0001",
    )
    parser.add_argument("--read-cycles","-rc", default="151", help="Readcycles (default: 301)")
    parser.add_argument("--index-cycles","-ic", default="10", help="Indexcycles (default: 8)")
    parser.add_argument("--output","-o", default="SampleSheet.csv",
        help="Output SampleSheet file (default: SampleSheet.csv)")

    args = parser.parse_args()

    log_file = Path(args.output).with_suffix(".log")

    if Path(args.output).exists():
        log(
            f"\nWARNING:\n"
            f"Output file already exists:\n"
            f"  {args.output}\n"
        )

        while True:

            answer = input(
                "Overwrite existing file? [y/n]: "
            ).strip().lower()

            if answer in {"y", "yes"}:
                log("Overwriting existing file.\n")
                break

            elif answer in {"n", "no"}:
                log("Operation cancelled.")
                sys.exit(0)

            else:
                log(
                    "Please answer with:\n"
                    "  y / yes\n"
                    "  n / no"
                )

    log("\n####################")
    log("Errors/Warnings/Info")
    log("####################\n")

    index_df = load_index_table(args.index_file)
    index_names = index_df["Index_Name"].tolist()
    index_lookup = index_df.set_index("Index_Name").to_dict(orient="index")

    df = parse_input_file(args.sample_config, index_names, args.start_index)
    
    validate_duplicate_indices_per_lane(df)
    
    write_samplesheet(args.output, args.run_name, df, index_lookup, args.read_cycles, args.index_cycles)
    
    unique_samples = df["sample_name"].nunique()
    
    log(f"INFO: Runname set to {args.run_name}")
    log(f"INFO: Read cycles set to {args.read_cycles}")
    log(f"INFO: Index cycles set to {args.index_cycles}")
    log(f"INFO: Total sample-lane combinations: {len(df)}")
    log(f"INFO: Unique samples: {unique_samples}")
    log("INFO: SampleSheet created successfully.")

    write_log_file(log_file)
    log(f"INFO: Log written to {log_file}")


if __name__ == "__main__":
    try:
        main()
    except SampleSheetError as e:
        log(f"ERROR: {e}")
        sys.exit(1)